In [8]:
%load_ext autoreload
%autoreload 2 

from numba import njit

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.cm as cm
from matplotlib.colors import LinearSegmentedColormap
import scipy
from kneed import KneeLocator

import neuro_lib as nlib

# SETTING SEED FOR REPRODUCIBILITY
np.random.seed(0)


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [4]:
@njit
def NEW_entropy_core(counts):
    total = np.sum(counts)
    if total == 0:
        return 0.0

    res = 0.0
    for i in range(len(counts)):
        c = counts[i]
        if c > 0:
            p = c / total
            res -= p * np.log2(p)
    return res

@njit
def NEW_bin_data_1d(x, n_bins, x_min, x_max):
    """
    Return bin count and number of bins with at least one count
    """
    counts = np.zeros(n_bins, dtype=np.int64)

    # degenerate case
    if x_max == x_min:
        return counts, 0

    width = (x_max - x_min) / n_bins

    for i in range(len(x)):
        idx = int((x[i] - x_min) / width)
        if idx >= n_bins:
            idx = n_bins - 1
        elif idx < 0:
            idx = 0
        counts[idx] += 1

    n_populated_bins = 0
    for i in range(n_bins):
        if counts[i] > 0:
            n_populated_bins += 1

    return counts, n_populated_bins

@njit
def NEW_entropy_absolute(X, n_bins):
    counts, n_populated_bins = NEW_bin_data_1d(X, n_bins, X.min(), X.max())
    return NEW_entropy_core(counts), n_populated_bins

@njit
def NEW_entropy_conditional(S, X, n_bins):
    """
    Returns entropy and number of bins with at least one count in the 2D grid
    """
    n_obs = len(S)

    s_min, s_max = S.min(), S.max()
    x_min, x_max = X.min(), X.max()

    s_width = (s_max - s_min) / n_bins if s_max > s_min else 1.0
    x_width = (x_max - x_min) / n_bins if x_max > x_min else 1.0

    counts_2d = np.zeros((n_bins, n_bins), dtype=np.int64)
    s_bin_totals = np.zeros(n_bins, dtype=np.int64)

    for j in range(n_obs):
        s_idx = int((S[j] - s_min) / s_width) if s_width > 0 else 0
        x_idx = int((X[j] - x_min) / x_width) if x_width > 0 else 0

        if s_idx >= n_bins:
            s_idx = n_bins - 1
        elif s_idx < 0:
            s_idx = 0

        if x_idx >= n_bins:
            x_idx = n_bins - 1
        elif x_idx < 0:
            x_idx = 0

        counts_2d[s_idx, x_idx] += 1
        s_bin_totals[s_idx] += 1

    h_cond = 0.0

    for i in range(n_bins):
        subset_size = s_bin_totals[i]
        if subset_size > 0:
            p_s = subset_size / n_obs
            h_cond += p_s * NEW_entropy_core(counts_2d[i, :])



    # count populated bins
    n_pop_bins = 0
    for i in range(n_bins):
        for j in range(n_bins):
            if counts_2d[i, j] > 0:
                n_pop_bins += 1


    return h_cond, n_pop_bins

@njit
def NEW_entropy_binning_1d(data, bin_number, which="absolute"):
    S = data[:, 0]
    X = data[:, 1]

    if which == "absolute":
        return NEW_entropy_absolute(X, bin_number)
    elif which == "conditional":
        return NEW_entropy_conditional(S, X, bin_number)
    else:
        raise ValueError("Invalid 'which' parameter. Use 'absolute' or 'conditional'.")

@njit
def NEW_mi_binning_2d(data, bin_number, bias_correction=False):
    Hx, x_populated_bins = NEW_entropy_binning_1d(data, bin_number, "absolute")
    Hx_given_s, joint_populated_bins = NEW_entropy_binning_1d(data, bin_number, "conditional")

    MI = Hx - Hx_given_s

    if bias_correction:
        N = data.shape[0]
        bias = (1 / (2 * N * np.log(2))) * (
            joint_populated_bins - x_populated_bins - (bin_number - 1)
        )
        return MI - bias

    return MI

In [5]:
np.sqrt(1000)

31.622776601683793